# Colab Demo — Turkish Legal RAG

Self-contained notebook for running the full RAG demo on Google Colab (T4 free tier) and exposing a public Gradio URL.

## Steps
1. **Runtime → Change runtime type → T4 GPU**
2. Run all cells top-to-bottom (`Shift+Enter`)
3. Upload `kaggle.json` when prompted (cell 3)
4. The final cell prints a `https://xxxxx.gradio.live` URL valid for ~72 hours

**Wall time:** ~10 minutes (dataset ~5 min + packages ~1 min + Trendyol-7B load ~4 min).

## 1. GPU check

In [1]:
!nvidia-smi | head -10

Mon Jun 15 18:16:25 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |


## 2. Dependencies (includes `torchao` for PEFT compatibility)

In [2]:
%%capture
!pip install -q gradio "sentence-transformers==3.3.1" "transformers==4.45.2" "peft==0.13.2" rank_bm25 faiss-cpu
!pip install -q -U bitsandbytes accelerate torchao
!pip install -q kaggle

## 3. Kaggle credentials

Run the cell, then click **Choose Files** and upload your `kaggle.json` (from your Kaggle account → API → Create New Token).

In [3]:
from google.colab import files
import os, shutil

uploaded = files.upload()
assert 'kaggle.json' in uploaded, 'kaggle.json yuklenmedi'

os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
shutil.move('kaggle.json', os.path.expanduser('~/.kaggle/kaggle.json'))
os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)
print('Kaggle credentials OK.')

Saving kaggle.json to kaggle.json
Kaggle credentials OK.


## 4. Download artifacts dataset (~3 GB, 3–5 min)

In [4]:
!kaggle datasets download -d hasanemreusta/turkish-legal-rag-system -p /content --unzip
!ls /content

Dataset URL: https://www.kaggle.com/datasets/hasanemreusta/turkish-legal-rag-system
License(s): apache-2.0
100% 4.66G/4.66G [04:41<00:00, 17.8MB/s]

data  ft_index_patch  sample_data  src


## 5. Workspace layout and FT FAISS merge

The Kaggle dataset ships the base-e5 FAISS index in `data/index_full/`. The fine-tuned index, when present as `ft_index_patch/`, is symlinked into the same directory so `Retriever` can resolve it.

In [5]:
import os
from pathlib import Path

# Dataset bazen klasor icinde, bazen flat extract olur — ikisini de dene
candidates = [Path('/content/turkish-legal-rag-system'), Path('/content')]
ROOT = None
for c in candidates:
    if (c / 'src').exists() and (c / 'data').exists():
        ROOT = c
        break
assert ROOT, 'Dataset bulunamadi. /content icerigini kontrol et.'
os.chdir(ROOT)
print('Working dir:', os.getcwd())

# FT FAISS dosyalarini ft_index_patch icinden bul ve data/index_full/'a symlink et
target = ROOT / 'data/index_full'
ft_patch_root = ROOT / 'ft_index_patch'
if ft_patch_root.exists():
    for root, dirs, files in os.walk(ft_patch_root):
        for fname in files:
            if 'data_models_e5-large-tr-legal' in fname and (fname.endswith('.index') or fname.endswith('.npy')):
                src = Path(root) / fname
                link = target / fname
                if not link.exists():
                    os.symlink(src, link)
                    print(f'linked: {fname}')

print('\nFinal data/index_full/:')
for f in sorted(target.iterdir()):
    print(' ', f.name)

# Sanity checks
assert (target / 'faiss_data_models_e5-large-tr-legal.index').exists(), 'FT FAISS yok!'
assert (ROOT / 'data/models/llm_adapter').exists(), 'LLM adapter yok!'
assert (ROOT / 'data/models/bge-reranker-tr-legal-v2').exists(), 'Reranker FT yok!'
print('\nTUM DOSYALAR HAZIR.')

Working dir: /content

Final data/index_full/:
  bm25.pkl
  embeddings_data_models_e5-large-tr-legal-v2.npy
  embeddings_data_models_e5-large-tr-legal.npy
  embeddings_intfloat_multilingual-e5-large.npy
  faiss_data_models_e5-large-tr-legal-v2.index
  faiss_data_models_e5-large-tr-legal.index
  faiss_intfloat_multilingual-e5-large.index
  meta.jsonl

TUM DOSYALAR HAZIR.


## 6. Write `src/demo/app.py`

The Gradio app is not included in the Kaggle dataset (which carries only data and `src/` library code). It is written out here so Colab can launch it without a separate `git clone`.

In [6]:
demo_dir = Path('src/demo')
demo_dir.mkdir(parents=True, exist_ok=True)
(demo_dir / '__init__.py').write_text('')

app_code = '''"""Gradio demo for Turkish Legal RAG."""
from __future__ import annotations
import logging, os
import gradio as gr
from src.pipeline.rag import RAGPipeline
from src.reranker import Reranker

log = logging.getLogger("demo")
def _env(name, default): return os.environ.get(name, default)

def build_pipeline():
    log.info("Loading reranker...")
    reranker = Reranker.load_v2(_env("DEMO_RERANKER_DIR", "data/models/bge-reranker-tr-legal-v2"))
    log.info("Building RAG (Trendyol-7B 4-bit + LoRA, ~3-4 min)...")
    rag = RAGPipeline.build(
        index_dir=_env("DEMO_INDEX_DIR", "data/index_full"),
        embed_model=_env("DEMO_EMBED_MODEL", "data/models/e5-large-tr-legal"),
        llm_backend="hf",
        llm_model=_env("DEMO_LLM_MODEL", "Trendyol/Trendyol-LLM-7B-chat-v4.1.0"),
        top_k=int(_env("DEMO_TOP_K", "5")),
        retrieval_mode="hybrid",
        reranker=reranker,
        candidate_k=int(_env("DEMO_CANDIDATE_K", "50")),
        adapter_path=_env("DEMO_ADAPTER_PATH", "data/models/llm_adapter"),
    )
    log.info("Pipeline ready.")
    return rag

def make_handler(rag):
    def ask(question):
        question = (question or "").strip()
        if not question:
            return "Lutfen bir soru girin.", "", ""
        try:
            out = rag.answer(question)
        except Exception as e:
            log.exception("Pipeline error")
            return f"Hata: {e}", "", ""
        sm, sf = [], []
        for i, s in enumerate(out["sources"], 1):
            head = f"**{i}. [{s['kanun_short']} m.{s['madde_no']}]** (score: {s['score']:.3f})"
            sm.append(head + "\\n\\n> " + s["text"][:300].replace("\\n", " ") + "...")
            sf.append(f"### {i}. [{s['kanun_short']} m.{s['madde_no']}]" + (f" - {s.get('baslik','')}" if s.get('baslik') else "") + f"\\n\\n{s['text']}\\n")
        return out["answer"], "\\n\\n".join(sm), "\\n\\n---\\n\\n".join(sf)
    return ask

def build_ui(rag):
    handler = make_handler(rag)
    with gr.Blocks(title="Turkish Legal RAG") as demo:
        gr.Markdown("# Turkish Legal RAG\\n\\nCitation-grounded Turkish legal question answering.")
        with gr.Row():
            with gr.Column(scale=2):
                q = gr.Textbox(label="Soru", placeholder="Orn: Cumhurbaskanina hakaret sucu cezasi nedir?", lines=3)
                submit = gr.Button("Cevapla", variant="primary")
                gr.Examples(examples=[
                    "Cumhurbaskanina hakaret sucu cezasi nedir?",
                    "Bosanma davasinda nafaka kosullari nelerdir?",
                    "Tuketici hangi hallerde sozlesmeden cayabilir?",
                    "Suc isleyen cocuk hakkindaki guvenlik tedbirleri nelerdir?",
                    "Isveren isciyi hakli sebeple hangi durumlarda derhal cikarabilir?",
                ], inputs=q)
            with gr.Column(scale=3):
                ans = gr.Textbox(label="Cevap (atif formatinda)", lines=8, show_copy_button=True)
        with gr.Tab("Kaynaklar (ozet)"):
            srs = gr.Markdown()
        with gr.Tab("Kaynaklar (tam metin)"):
            srf = gr.Markdown()
        submit.click(handler, inputs=q, outputs=[ans, srs, srf])
        q.submit(handler, inputs=q, outputs=[ans, srs, srf])
    return demo

def main():
    logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s | %(message)s")
    rag = build_pipeline()
    demo = build_ui(rag)
    demo.queue().launch(share=True, server_name="0.0.0.0", server_port=7860)

if __name__ == "__main__":
    main()
'''
(demo_dir / 'app.py').write_text(app_code, encoding='utf-8')
print('src/demo/app.py written.')
!ls src/demo/

src/demo/app.py written.
app.py	__init__.py


## 7. Launch the Gradio app

Takes ~4–5 minutes (4-bit Trendyol-7B download + LoRA merge). When ready, the console prints:

```
Running on public URL: https://xxxxxxxx.gradio.live
```

The URL is valid for ~72 hours.

In [7]:
!python -m src.demo.app

2026-06-15 18:23:44,205 INFO demo | Loading reranker...
2026-06-15 18:23:52,701 INFO numexpr.utils | NumExpr defaulting to 2 threads.
2026-06-15 18:23:53,793 WARNING torchao | Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
2026-06-15 18:23:53,797 WARNING torchao | Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
2026-06-15 18:24:02,295 INFO datasets | TensorFlow version 2.20.0 available.
2026-06-15 18:24:02,296 INFO datasets | JAX version 0.7.2 available.
2026-06-15 18:24:07,652 INFO reranker | Loading v2 reranker: base=BAAI/bge-reranker-v2-m3 + adapter=data/models/bge-reranker-tr-legal-v2/lora_adapter (device=cuda)
2026-06-15 18:24:07,945 INFO httpx | HTTP Request: HEAD https:

In [ ]:
!pip install -q --force-reinstall \
  "sentence-transformers==3.3.1" \
  "transformers==4.45.2" \
  "peft==0.13.2"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.7/58.7 kB 5.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 6.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.8/268.8 kB 25.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 117.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.7/320.7 kB 34.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 50.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 104.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.2/100.2 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 807.9/807.9 kB 64.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━